# Part 4 — LLM-Powered Intelligent System with Production Guardrails

## Applied AI & Machine Learning Essentials — Capstone Project

### Industry-Level Production Implementation

---

### Project Objective

The objective of Part 4 is to design and implement a production-ready Large Language Model (LLM) powered intelligent scoring system capable of processing structured tabular records using prompt engineering, JSON schema validation, and robust production guardrails.

This implementation focuses on building a secure, reliable, and scalable AI pipeline that performs automated batch scoring while ensuring deterministic outputs, structured JSON responses, input validation, prompt safety, error handling, and deployment-ready architecture.

---

### Key Objectives

- Develop a reusable LLM API client using environment variables for secure authentication.
- Implement production-grade PII guardrails to prevent sensitive information from reaching the LLM.
- Design an evaluation rubric and structured system prompt with JSON schema validation.
- Process multiple real records from the cleaned dataset using batch inference.
- Compare deterministic and creative model behaviour using different temperature settings.
- Validate every LLM response using JSON Schema and fallback mechanisms.
- Log errors safely and ensure robust exception handling.
- Demonstrate an end-to-end production workflow suitable for real-world deployment.



In [17]:
import os
print(repr(os.environ.get("LLM_API_KEY")))

'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'


In [19]:
import os
os.environ["LLM_API_KEY"] = "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"

In [20]:
from google.colab import userdata
os.environ["LLM_API_KEY"] = userdata.get("LLM_API_KEY")

In [45]:
import os
import requests

api_key = os.environ.get("LLM_API_KEY")
print(f"Key present: {bool(api_key)}")
print(f"Key preview: {api_key[:15] if api_key else 'NOT SET'}...")
print()

# Fetch full model list from your account
response = requests.get(
    "https://openrouter.ai/api/v1/models",
    headers={"Authorization": f"Bearer {api_key}"}
)

print(f"Status: {response.status_code}")

if response.status_code == 200:
    models = response.json().get("data", [])
    print(f"Total models available: {len(models)}")
    print()
    print("ALL FREE MODELS (prompt cost = 0):")
    print("-" * 60)
    count = 0
    for m in models:
        pricing = m.get("pricing", {})
        try:
            prompt_cost = float(pricing.get("prompt", 1))
        except:
            prompt_cost = 1
        if prompt_cost == 0:
            print(m["id"])
            count += 1
    print("-" * 60)
    print(f"Total free models found: {count}")
else:
    print(f"Error fetching models: {response.text}")

Key present: True
Key preview: sk-or-v1-30700a...

Status: 200
Total models available: 343

ALL FREE MODELS (prompt cost = 0):
------------------------------------------------------------
tencent/hy3:free
poolside/laguna-xs-2.1:free
cohere/north-mini-code:free
nvidia/nemotron-3.5-content-safety:free
nvidia/nemotron-3-ultra-550b-a55b:free
nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free
poolside/laguna-m.1:free
google/gemma-4-26b-a4b-it:free
google/gemma-4-31b-it:free
google/lyria-3-pro-preview
google/lyria-3-clip-preview
nvidia/nemotron-3-super-120b-a12b:free
openrouter/free
nvidia/nemotron-3-nano-30b-a3b:free
nvidia/nemotron-nano-12b-v2-vl:free
qwen/qwen3-next-80b-a3b-instruct:free
nvidia/nemotron-nano-9b-v2:free
openai/gpt-oss-120b:free
openai/gpt-oss-20b:free
qwen/qwen3-coder:free
cognitivecomputations/dolphin-mistral-24b-venice-edition:free
meta-llama/llama-3.3-70b-instruct:free
meta-llama/llama-3.2-3b-instruct:free
nousresearch/hermes-3-llama-3.1-405b:free
----------------------

In [46]:
import os
import requests

api_key = os.environ.get("LLM_API_KEY")

models_to_try = [
    "meta-llama/llama-3.3-70b-instruct:free",
    "meta-llama/llama-3.2-3b-instruct:free",
    "google/gemma-4-31b-it:free",
    "qwen/qwen3-coder:free",
    "openai/gpt-oss-20b:free",
]

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

working_model = None

for model in models_to_try:
    print(f"Testing: {model}")
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user",   "content": "Reply with only the word: hello"}
        ],
        "temperature": 0.0,
        "max_tokens": 10
    }
    try:
        response = requests.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers=headers,
            json=payload,
            timeout=30
        )
        if response.status_code == 200:
            content = response.json()["choices"][0]["message"]["content"]
            print(f"✓ WORKS — Reply: {content!r}")
            working_model = model
            break
        else:
            print(f"✗ Status {response.status_code}: {response.text[:100]}")
    except Exception as e:
        print(f"✗ Exception: {e}")
    print("-" * 50)

if working_model:
    print(f"\n✓ USE THIS MODEL: '{working_model}'")
else:
    print("\n✗ None worked — paste full output here")

Testing: meta-llama/llama-3.3-70b-instruct:free
✗ Status 429: {"error":{"message":"Provider returned error","code":429,"metadata":{"raw":"meta-llama/llama-3.3-70b
--------------------------------------------------
Testing: meta-llama/llama-3.2-3b-instruct:free
✗ Status 429: {"error":{"message":"Provider returned error","code":429,"metadata":{"raw":"meta-llama/llama-3.2-3b-
--------------------------------------------------
Testing: google/gemma-4-31b-it:free
✗ Status 429: {"error":{"message":"Provider returned error","code":429,"metadata":{"raw":"google/gemma-4-31b-it:fr
--------------------------------------------------
Testing: qwen/qwen3-coder:free
✗ Status 429: {"error":{"message":"Provider returned error","code":429,"metadata":{"raw":"qwen/qwen3-coder:free is
--------------------------------------------------
Testing: openai/gpt-oss-20b:free
✓ WORKS — Reply: 'hello'

✓ USE THIS MODEL: 'openai/gpt-oss-20b:free'


In [49]:
"""
Part 4 — LLM-Powered Intelligent System
Track B: Tabular Record Batch Scoring
Step 1: call_llm() reusable client + test call
"""

import os
import requests
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
logger = logging.getLogger(__name__)

# ── Correct working model ─────────────────────────────────────────────────────
LLM_URL   = "https://openrouter.ai/api/v1/chat/completions"
LLM_MODEL = "openai/gpt-oss-20b:free"


def call_llm(
    system_prompt: str,
    user_prompt: str,
    temperature: float = 0.0,
    max_tokens: int = 512
):
    """
    Reusable LLM API client.

    Parameters
    ----------
    system_prompt : str   Role/instruction context for the model.
    user_prompt   : str   The actual user-facing input.
    temperature   : float Sampling temperature (0 = deterministic).
    max_tokens    : int   Maximum tokens in the response.

    Returns
    -------
    str | None  Model response text, or None on failure.
    """
    api_key = os.environ.get("LLM_API_KEY")
    if not api_key:
        logger.error("LLM_API_KEY environment variable is not set.")
        return None

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": LLM_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt}
        ],
        "temperature": temperature,
        "max_tokens":  max_tokens
    }

    try:
        response = requests.post(
            LLM_URL, headers=headers, json=payload, timeout=30
        )

        if response.status_code != 200:
            print(f"LLM API error: status code {response.status_code}")
            print(f"Response body : {response.text}")
            return None

        content = response.json()["choices"][0]["message"]["content"]
        logger.info("LLM call successful. Tokens used: %s",
                    response.json().get("usage", {}).get("total_tokens", "N/A"))
        return content

    except requests.exceptions.Timeout:
        logger.error("LLM API request timed out.")
        return None
    except requests.exceptions.RequestException as exc:
        logger.error("LLM API request failed: %s", exc)
        return None
    except (KeyError, IndexError) as exc:
        logger.error("Unexpected response structure: %s", exc)
        return None


# ── Test call ─────────────────────────────────────────────────────────────────
print("=" * 60)
print("STEP 1 — call_llm() Test Call")
print("=" * 60)

test_system = "You are a helpful assistant."
test_user   = "Reply with only the word: hello"

result = call_llm(
    system_prompt=test_system,
    user_prompt=test_user,
    temperature=0.0,
    max_tokens=10
)

print(f"\nTest prompt : {test_user!r}")
print(f"LLM response: {result!r}")

if result and "hello" in result.strip().lower():
    print("\n✓ Step 1 PASSED — API connection confirmed.")
else:
    print("\n✗ Step 1 FAILED — check API key and model name.")

STEP 1 — call_llm() Test Call

Test prompt : 'Reply with only the word: hello'
LLM response: 'hello'

✓ Step 1 PASSED — API connection confirmed.


In [50]:
"""
Part 4 — Track B
Step 2: PII Guardrail — has_pii() function + demo calls
"""

import re


def has_pii(text: str) -> bool:
    """
    Checks whether the input text contains personally identifiable
    information (PII) — specifically email addresses or phone numbers.

    Parameters
    ----------
    text : str  The user input string to check.

    Returns
    -------
    bool  True if PII is detected, False otherwise.
    """
    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'
    return bool(re.search(email_pattern, text) or re.search(phone_pattern, text))


def safe_call_llm(
    system_prompt: str,
    user_prompt: str,
    temperature: float = 0.0,
    max_tokens: int = 512
):
    """
    Wrapper around call_llm() that runs the PII guardrail
    before every API call.
    """
    if has_pii(user_prompt):
        print("Input blocked: PII detected.")
        return None

    return call_llm(
        system_prompt=system_prompt,
        user_prompt=user_prompt,
        temperature=temperature,
        max_tokens=max_tokens
    )


# ── Demo Call 1 — BLOCKED ─────────────────────────────────────────────────────
print("=" * 60)
print("STEP 2 — PII Guardrail Demo")
print("=" * 60)

print("\n--- Demo Call 1: Input WITH email address (should be BLOCKED) ---")

blocked_input = (
    "Please score this record. "
    "Contact: john.doe@example.com. "
    "Age: 35, hours_per_week: 40."
)

print(f"Input : {blocked_input!r}")

result_blocked = safe_call_llm(
    system_prompt="You are a scoring assistant.",
    user_prompt=blocked_input
)

print(f"Result: {result_blocked}")


# ── Demo Call 2 — PASSED ──────────────────────────────────────────────────────
print("\n--- Demo Call 2: Input WITHOUT PII (should PROCEED) ---")

clean_input = "Age: 35, hours_per_week: 40, education: Bachelors."

print(f"Input : {clean_input!r}")

result_clean = safe_call_llm(
    system_prompt="You are a helpful assistant.",
    user_prompt=clean_input,
    temperature=0.0,
    max_tokens=20
)

print(f"Result: {result_clean!r}")


# ── Verification Summary ──────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2 — Verification Summary")
print("=" * 60)

blocked_ok = result_blocked is None
passed_ok  = result_clean is not None

print(f"Demo 1 (email input) blocked correctly : {blocked_ok}")
print(f"Demo 2 (clean input) passed through    : {passed_ok}")

if blocked_ok and passed_ok:
    print("\n✓ Step 2 PASSED — PII guardrail working correctly.")
else:
    print("\n✗ Step 2 FAILED — review output above.")

STEP 2 — PII Guardrail Demo

--- Demo Call 1: Input WITH email address (should be BLOCKED) ---
Input : 'Please score this record. Contact: john.doe@example.com. Age: 35, hours_per_week: 40.'
Input blocked: PII detected.
Result: None

--- Demo Call 2: Input WITHOUT PII (should PROCEED) ---
Input : 'Age: 35, hours_per_week: 40, education: Bachelors.'
Result: '**Quick Snapshot**\n\n| Factor | Detail | Implication |\n|--------|--------|-------------|\n| **Age** | 35 | Mid‑career stage; many people are solidifying their professional identity and looking for growth or transition. |\n| **Hours per week** | 40 | Full‑time commitment; you likely have a stable routine but may have limited spare time for side projects or learning. |\n| **Education** | Bachelor’s degree | Strong foundation; many roles require at least a bachelor’s, but advanced positions often favor additional credentials or experience. |\n\n---\n\n## 1. What You’re Likely Doing\n\n| Typical Path | What It Looks Like |\n|---------

In [56]:
income_schema = {
    "type": "object",
    "required": [
        "income_score",
        "income_class",
        "prediction",
        "key_factors",
        "recommendation"
    ],
    "properties": {
        "income_score": {
            "type": "number"
        },
        "income_class": {
            "type": "string"
        },
        "prediction": {
            "type": "string"
        },
        "key_factors": {
            "type": "string"
        },
        "recommendation": {
            "type": "string"
        }
    }
}

In [71]:
system_prompt = """
You are an AI income prediction assistant.

Your task is to analyze Adult Income Dataset records
and predict whether a person earns <=50K or >50K.

Instructions:
- Analyze the given customer/person attributes.
- Generate income_score between 0 and 100.
- Return ONLY valid JSON that strictly conforms to the provided JSON schema.
- Do not provide explanations or any text outside the JSON object.

JSON Schema:
```json
{
    "type": "object",
    "required": [
        "income_score",
        "income_class",
        "prediction",
        "key_factors",
        "recommendation"
    ],
    "properties": {
        "income_score": {
            "type": "number",
            "description": "A numerical score representing the likelihood of income >50K, between 0 and 100."
        },
        "income_class": {
            "type": "string",
            "enum": ["<=50K", ">50K"],
            "description": "The predicted income class."
        },
        "prediction": {
            "type": "string",
            "description": "A concise prediction statement."
        },
        "key_factors": {
            "type": "string",
            "description": "Key factors influencing the prediction."
        },
        "recommendation": {
            "type": "string",
            "description": "A recommendation based on the prediction."
        }
    }
}
```

Consider these factors:
- Age
- Education
- Education number
- Workclass
- Occupation
- Hours per week
- Capital gain
- Capital loss
- Marital status

Example Input:

{
 "age": 39,
 "education": "Bachelors",
 "education_num": 13,
 "workclass": "State-gov",
 "occupation": "Adm-clerical",
 "hours_per_week": 40,
 "capital_gain": 2174,
 "capital_loss": 0
}

Example Output (adhering to schema):
```json
{
  "income_score": 75,
  "income_class": ">50K",
  "prediction": "This individual is predicted to earn more than 50K annually.",
  "key_factors": "High education level (Bachelors), state-government workclass, and significant capital gains.",
  "recommendation": "Consider opportunities for career advancement given strong profile."
}
```
"""

In [59]:
import pandas as pd
import json

# Load cleaned dataset
df = pd.read_csv("cleaned_data.csv")

# Select 3 real records
sample_records = df.head(3)

# Convert to JSON format
records_json = sample_records.to_dict(orient="records")

print(json.dumps(records_json, indent=4))

[
    {
        "age": 39,
        "workclass": "State-gov",
        "fnlwgt": 77516,
        "education": "Bachelors",
        "education_num": 13,
        "marital_status": "Never-married",
        "occupation": "Adm-clerical",
        "relationship": "Not-in-family",
        "race": "White",
        "sex": "Male",
        "capital_gain": 2174,
        "capital_loss": 0,
        "hours_per_week": 40,
        "native_country": "United-States",
        "income": 0
    },
    {
        "age": 50,
        "workclass": "Self-emp-not-inc",
        "fnlwgt": 83311,
        "education": "Bachelors",
        "education_num": 13,
        "marital_status": "Married-civ-spouse",
        "occupation": "Exec-managerial",
        "relationship": "Husband",
        "race": "White",
        "sex": "Male",
        "capital_gain": 0,
        "capital_loss": 0,
        "hours_per_week": 13,
        "native_country": "United-States",
        "income": 0
    },
    {
        "age": 38,
        "workclass"

In [64]:
import json
import jsonschema
import logging

In [66]:
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s: %(message)s"
)

In [67]:
fallback_response = {
    "income_score": None,
    "income_class": None,
    "prediction": None,
    "key_factors": None,
    "recommendation": None
}

In [74]:
def batch_score_records(records):

    results = []

    for index, record in enumerate(records, start=1):

        print("\n" + "=" * 60)
        print(f"Processing Record {index}")
        print("=" * 60)

        # Convert record into JSON string
        user_prompt = json.dumps(record)

        print("Input:")
        print(user_prompt)


        # ------------------------------------------------
        # Step 1: PII Guardrail
        # ------------------------------------------------
        if has_pii(user_prompt):

            print("Input blocked: PII detected.")

            results.append({
                "input": record,
                "output": fallback_response,
                "status": "BLOCKED"
            })

            continue


        # ------------------------------------------------
        # Step 2: LLM Call
        # ------------------------------------------------
        raw_response = call_llm(
            system_prompt=system_prompt,
            user_prompt=user_prompt,
            temperature=0.0,
            max_tokens=512  # Increased max_tokens to allow for complete response
        )

        print("\nRaw LLM Response:")
        print(raw_response)

        if raw_response is None:
            logging.error("LLM call returned None. Using fallback response.")
            parsed_response = fallback_response
            validation_status = "LLM_FAILED"
        else:
            # ------------------------------------------------
            # Step 3: Strip Response (removing markdown code block wrapper)
            # ------------------------------------------------
            cleaned_response = raw_response.strip()
            if cleaned_response.startswith('```json') and cleaned_response.endswith('```'):
                cleaned_response = cleaned_response[7:-3].strip()


            # ------------------------------------------------
            # Step 4: JSON Parse + Schema Validation
            # ------------------------------------------------
            try:

                parsed_response = json.loads(cleaned_response)


                jsonschema.validate(
                    instance=parsed_response,
                    schema=income_schema
                )


                validation_status = "PASSED"


            except Exception as e:

                logging.error(
                    f"Validation failed: {e}"
                )

                parsed_response = fallback_response
                validation_status = "FAILED"



        print("\nValidation:")
        print(validation_status)


        # ------------------------------------------------
        # Step 5: Store Result
        # ------------------------------------------------
        results.append({

            "input": record,

            "raw_response": raw_response,

            "validated_output": parsed_response,

            "status": validation_status

        })


    return results

In [75]:
batch_results = batch_score_records(records_json)


Processing Record 1
Input:
{"age": 39, "workclass": "State-gov", "fnlwgt": 77516, "education": "Bachelors", "education_num": 13, "marital_status": "Never-married", "occupation": "Adm-clerical", "relationship": "Not-in-family", "race": "White", "sex": "Male", "capital_gain": 2174, "capital_loss": 0, "hours_per_week": 40, "native_country": "United-States", "income": 0}

Raw LLM Response:
```json
{
  "income_score": 70,
  "income_class": ">50K",
  "prediction": "This individual is predicted to earn more than 50K annually.",
  "key_factors": "Bachelor's education, state-government workclass, and significant capital gains contribute to a higher income likelihood.",
  "recommendation": "Leverage current position for further career growth and consider additional certifications to maximize earning potential."
}
```

Validation:
PASSED

Processing Record 2
Input:
{"age": 50, "workclass": "Self-emp-not-inc", "fnlwgt": 83311, "education": "Bachelors", "education_num": 13, "marital_status": "Marr

In [78]:
def temperature_ab_test(records):

    ab_results = []

    temperatures = [0.0, 0.7]

    for temp in temperatures:

        print("\n" + "=" * 60)
        print(f"Running Temperature = {temp}")
        print("=" * 60)


        for index, record in enumerate(records, start=1):

            print(f"\nRecord {index}")

            user_prompt = json.dumps(record)

            print("Input:")
            print(user_prompt)


            # ----------------------------------------
            # PII Guardrail before every LLM call
            # ----------------------------------------
            if has_pii(user_prompt):

                print("Input blocked: PII detected.")

                ab_results.append({
                    "record": index,
                    "temperature": temp,
                    "status": "BLOCKED",
                    "output": fallback_response
                })

                continue


            # ----------------------------------------
            # LLM Call
            # ----------------------------------------
            raw_response = call_llm(
                system_prompt=system_prompt,
                user_prompt=user_prompt,
                temperature=temp,
                max_tokens=512 # Increased max_tokens to prevent truncation
            )


            print("\nRaw Response:")
            print(raw_response)


            # ----------------------------------------
            # Parse + Validate
            # ----------------------------------------
            if raw_response is None:
                logging.error("LLM call returned None. Using fallback response.")
                parsed_response = fallback_response
                status = "LLM_FAILED"
            else:
                cleaned_response = raw_response.strip()
                # Remove markdown code block wrapper if present
                if cleaned_response.startswith('```json') and cleaned_response.endswith('```'):
                    cleaned_response = cleaned_response[7:-3].strip()

                try:
                    parsed_response = json.loads(
                        cleaned_response
                    )


                    jsonschema.validate(
                        instance=parsed_response,
                        schema=income_schema
                    )


                    status = "PASSED"


                except Exception as e:

                    logging.error(
                        f"Validation failed: {e}"
                    )

                    parsed_response = fallback_response

                    status = "FAILED"



            print("\nValidation:")
            print(status)


            ab_results.append({

                "record": index,

                "temperature": temp,

                "output": parsed_response,

                "status": status

            })


    return ab_results

In [79]:
ab_results = temperature_ab_test(records_json)


Running Temperature = 0.0

Record 1
Input:
{"age": 39, "workclass": "State-gov", "fnlwgt": 77516, "education": "Bachelors", "education_num": 13, "marital_status": "Never-married", "occupation": "Adm-clerical", "relationship": "Not-in-family", "race": "White", "sex": "Male", "capital_gain": 2174, "capital_loss": 0, "hours_per_week": 40, "native_country": "United-States", "income": 0}

Raw Response:
{"income_score":70,"income_class":">50K","prediction":"This individual is predicted to earn more than 50K annually.","key_factors":"Bachelor’s education, state‑government employment, 40 hours per week, and significant capital gains.","recommendation":"Leverage current strengths for further career advancement and consider additional income‑generating opportunities."}

Validation:
PASSED

Record 2
Input:
{"age": 50, "workclass": "Self-emp-not-inc", "fnlwgt": 83311, "education": "Bachelors", "education_num": 13, "marital_status": "Married-civ-spouse", "occupation": "Exec-managerial", "relations

In [80]:
def create_demo_table(batch_results):

    demo_table = []

    for result in batch_results:

        demo_table.append({

            "Input": result["input"],

            "LLM Output": result.get(
                "raw_response",
                None
            ),

            "Valid JSON": (
                True
                if result["status"] == "PASSED"
                else False
            ),

            "Status": result["status"]

        })

    return demo_table

In [82]:
demo_results = create_demo_table(batch_results)

In [83]:
import pandas as pd

demo_df = pd.DataFrame(demo_results)

print("\n" + "="*80)
print("STEP 7 — END-TO-END DEMO TABLE")
print("="*80)

print(demo_df.to_string(index=False))


STEP 7 — END-TO-END DEMO TABLE
                                                                                                                                                                                                                                                                                                                                                       Input                                                                                                                                                                                                                                                                                                                                                                                                                                                                  LLM Output  Valid JSON Status
      {'age': 39, 'workclass': 'State-gov', 'fnlwgt': 77516, 'education': 'Bachelors', 'education_num': 13, 'marital_status': 'Never-married

In [85]:
import os
import jsonschema
from jsonschema import ValidationError


print("="*60)
print("FINAL ACCEPTANCE CHECK")
print("="*60)


# 1. API Key Check
if os.getenv("LLM_API_KEY"):
    print("✅ API key stored in environment variable")
else:
    print("❌ API key missing")


# 2. call_llm Test
print("\n1. call_llm Test")

response = call_llm(
    system_prompt="Reply with only one word.",
    user_prompt="hello",
    temperature=0
)

print("Response:", response)


# 3. PII Guardrail Test
print("\n2. PII Guardrail Test")

print("Email Test:")

if has_pii("Contact: john.doe@example.com"):
    print("✅ Blocked correctly")
else:
    print("❌ Failed")


print("Clean Test:")

if not has_pii("Age:39 Education:Bachelors"):
    print("✅ Passed correctly")
else:
    print("❌ Failed")


# 4. Schema Validation Test
print("\n3. Schema Validation Test")

test_output = {
    "income_score":70,
    "income_class":">50K",
    "prediction":"High income probability",
    "key_factors":"Education and job",
    "recommendation":"Improve skills"
}


try:
    jsonschema.validate(
        test_output,
        income_schema
    )

    print("✅ JSON validation passed")

except ValidationError as e:
    print("❌ Validation Error:", e.message)


# 5. Batch Scoring Test
print("\n4. Batch Scoring Test")

results = batch_score_records(records_json)

for i,r in enumerate(results,1):
    print(
        f"Record {i}: {r['status']}"
    )


# 6. Temperature A/B Test
print("\n5. Temperature A/B Test")

ab_results = temperature_ab_test(records_json)

for r in ab_results:
    print(
        "Record:",
        r["record"],
        "Temp:",
        r["temperature"],
        r["status"]
    )


print("\n🎉 Acceptance Check Completed")

FINAL ACCEPTANCE CHECK
✅ API key stored in environment variable

1. call_llm Test
Response: Hi!

2. PII Guardrail Test
Email Test:
✅ Blocked correctly
Clean Test:
✅ Passed correctly

3. Schema Validation Test
✅ JSON validation passed

4. Batch Scoring Test

Processing Record 1
Input:
{"age": 39, "workclass": "State-gov", "fnlwgt": 77516, "education": "Bachelors", "education_num": 13, "marital_status": "Never-married", "occupation": "Adm-clerical", "relationship": "Not-in-family", "race": "White", "sex": "Male", "capital_gain": 2174, "capital_loss": 0, "hours_per_week": 40, "native_country": "United-States", "income": 0}

Raw LLM Response:
```json
{
  "income_score": 22,
  "income_class": "<=50K",
  "prediction": "This individual is predicted to earn 50K or less annually.",
  "key_factors": "Moderate age, standard education (Bachelors), and no significant capital gains or losses.",
  "recommendation": "Consider pursuing higher education or specialized training to increase earning potent